# Module 08: Object Serialization with Pickle
## Notebook 01: Pickle Fundamentals, Byte Streams, and Protocol Evolution

In Python and machine learning engineering, **Serialization** (pickling) is the process of translating in-memory Python object hierarchies into persistent byte streams. **Deserialization** (unpickling) reconstructs the original object state from bytes. Python's built-in `pickle` module is the foundational serialization engine underlying model persistence, caching, and inter-process communication.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand the mechanics of **Pickling** and **Unpickling** in memory and on disk.
2. Master the core API: `pickle.dump()`, `pickle.load()`, `pickle.dumps()`, and `pickle.loads()`.
3. Identify which Python types can be pickled natively and understand limitations (lambdas, generators, open handles).
4. Navigate the evolution of **Pickle Protocols (0 to 5)** and understand their performance characteristics.
5. **Advanced:** Utilize **Pickle Protocol 5 Out-of-Band Buffers** for zero-copy serialization of massive numeric arrays.
6. **Advanced:** Serialize and restore complete Scikit-Learn machine learning pipelines with metadata.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print(f"Python Pickle Highest Protocol Supported: {pickle.HIGHEST_PROTOCOL}")
print(f"Default Pickle Protocol in this Python runtime: {pickle.DEFAULT_PROTOCOL}")

### 1. In-Memory vs. File-Based Serialization
The `pickle` API provides two complementary pairs of functions:
- **`pickle.dumps(obj)`:** Serializes Python object `obj` to a `bytes` object in memory.
- **`pickle.loads(bytes_obj)`:** Reconstructs the Python object from in-memory `bytes`.
- **`pickle.dump(obj, file)`:** Serializes directly to an open binary file stream (`mode='wb'`).
- **`pickle.load(file)`:** Reconstructs directly from an open binary file stream (`mode='rb'`).

In [ ]:
# 1. In-Memory Serialization with dumps/loads
sample_data = {
    "experiment_id": "EXP-2026-904",
    "hyperparameters": {"learning_rate": 0.001, "batch_size": 64, "optimizer": "AdamW"},
    "metrics": [0.945, 0.952, 0.961],
    "converged": True
}

# Serialize to raw byte stream
serialized_bytes = pickle.dumps(sample_data)
print(f"Serialized Byte Stream Type: {type(serialized_bytes)}")
print(f"Byte Length: {len(serialized_bytes)} bytes")
print(f"First 30 raw bytes: {serialized_bytes[:30]}")

# Deserialize back to Python dictionary
restored_data = pickle.loads(serialized_bytes)
print(f"\nRestored Object: {restored_data}")
assert restored_data == sample_data, "Deserialized data does not match original!"

### 2. File-Based Model Persistence
When persisting models to disk, always open files in **binary mode** (`'wb'` for writing, `'rb'` for reading). Opening in text mode (`'w'` or `'r'`) causes encoding corruption on newline characters.

In [ ]:
# Train a simple Scikit-Learn classification pipeline
X_dummy = np.random.randn(200, 4)
y_dummy = (X_dummy[:, 0] + X_dummy[:, 1] > 0).astype(int)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])
pipeline.fit(X_dummy, y_dummy)

# Package model with metadata envelope
model_bundle = {
    "model": pipeline,
    "version": "1.0.4",
    "feature_names": ["feat_1", "feat_2", "feat_3", "feat_4"],
    "train_accuracy": float(pipeline.score(X_dummy, y_dummy))
}

# Save to disk
bundle_path = "model_bundle.pkl"
with open(bundle_path, "wb") as f:
    pickle.dump(model_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved model bundle to {bundle_path} ({os.path.getsize(bundle_path)} bytes)")

# Load from disk and verify inference
with open(bundle_path, "rb") as f:
    loaded_bundle = pickle.load(f)

test_sample = np.array([[0.5, -0.2, 1.1, -0.8]])
orig_pred = pipeline.predict_proba(test_sample)
loaded_pred = loaded_bundle["model"].predict_proba(test_sample)

np.testing.assert_allclose(orig_pred, loaded_pred)
print(f"Verification Successful: Loaded model probabilities match original ({loaded_pred[0].tolist()})")

# Clean up temporary artifact
if os.path.exists(bundle_path):
    os.remove(bundle_path)

### 3. Pickle Protocol Evolution (Protocols 0 through 5)
Pickle has evolved across Python versions to improve speed, memory efficiency, and object support:
- **Protocol 0:** Original human-readable ASCII protocol (slow, verbose).
- **Protocol 1 & 2:** Compact binary formats introduced in early Python.
- **Protocol 3:** Introduced in Python 3.0; native `bytes` object support.
- **Protocol 4:** Python 3.4+ default; added support for massive 64-bit objects (>4GB), `__slots__`, and optimized code-paths.
- **Protocol 5:** Python 3.8+; introduced **Out-of-Band Buffers (`PickleBuffer`)** for zero-copy memory transfers of contiguous array buffers (crucial for high-performance ML).

In [ ]:
# Benchmark size and speed across protocols on a numeric matrix
test_matrix = np.random.randn(500, 500) # 250,000 float64 elements (~2 MB)

print(f"{'Protocol':<10} | {'Byte Size':<12} | {'Relative Ratio':<15}")
print("-" * 42)

sizes = {}
for proto in range(pickle.HIGHEST_PROTOCOL + 1):
    payload = pickle.dumps(test_matrix, protocol=proto)
    sizes[proto] = len(payload)
    print(f"Protocol {proto:<2} | {len(payload):<12,d} bytes | {len(payload) / sizes[0]:.2f}x")

### 4. Complex Application: Zero-Copy Serialization with Protocol 5 Out-of-Band Buffers
In distributed machine learning (e.g. Ray, Dask, multiprocessing), passing large arrays through standard pickle creates duplicate memory copies:
1. `test_array` (200 MB) $\to$ copied into serialized `bytes` buffer (200 MB) $\to$ copied back upon load.
**Protocol 5 Out-of-Band Buffers** solve this:
- Extracts memory buffers into external views without copying them into the main pickle bytecode stream.
- The bytecode stream stores only lightweight structural metadata; the heavy buffers are transferred zero-copy via shared memory.

In [ ]:
# Create large multi-gigabyte simulation array
large_array = np.arange(1_000_000, dtype=np.float64) # 8 MB contiguous buffer

out_of_band_buffers = []

def buffer_callback(buffer):
    # Intercept raw memory buffer without duplicating into pickle stream
    out_of_band_buffers.append(buffer)

# 1. Serialize using Protocol 5 and buffer_callback
header_bytes = pickle.dumps(
    large_array,
    protocol=5,
    buffer_callback=buffer_callback
)

print(f"Original Array Size:         {large_array.nbytes:,} bytes")
print(f"Pickle Header Metadata Size: {len(header_bytes):,} bytes (Extremely compact!)")
print(f"Captured Out-of-Band Buffers: {len(out_of_band_buffers)} buffer(s)")

# 2. Deserialize reconstructing zero-copy from the external buffers
restored_array = pickle.loads(header_bytes, buffers=out_of_band_buffers)
np.testing.assert_array_equal(large_array, restored_array)
print("SUCCESS: Zero-copy reconstructed array perfectly matches source memory!")